In [ ]:
import requests
import ollama
import json
from bs4 import BeautifulSoup
# from openai import OpenAI
from IPython.display import Markdown, display, update_display
import re

In [7]:
MODEL = 'gemma3:4b'
# ollama_model = ollama(base_url="https://localhost:11434/v1",api_key="ollama")

In [8]:

class Website:
    def __init__(self,url):
        self.url = url
        response = requests.get(url)
        self.body = response.content
        soup = BeautifulSoup(self.body,'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator='\n',strip=True)
        else:
            self.text=""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents: \n{self.text}\n"

In [9]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON object format as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [10]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms o f Service, Privacy, email links \n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [11]:
def get_links(url):
    website = Website(url)
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role":"system","content":link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
        ]
    )
    result = response["message"]["content"]

    cleaned_result = re.sub(r"^```(?:json)?\n|\n```$", "", result.strip())

    return json.loads(cleaned_result)

In [12]:
get_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'},
  {'type': 'models', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces', 'url': 'https://huggingface.co/spaces'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'models', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs/transformers'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs/diffusers'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs/safetensors'},
  {'type': 'documentation',
   'url': 'https://huggingface.co/docs/huggingface_hub'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs/tokenizers'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs/trl'},
  {'

In [13]:
def get_all_details(url):
    result = "Landing Page"
    result += Website(url).get_contents()
    links = get_links(url)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [15]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."


In [16]:
def get_brochure_user_prompt(company_name,url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5000] # Truncate if more than 5000 characters
    return user_prompt

In [26]:

from IPython.display import display, Markdown

def create_stream_brochure(company_name, url):
    stream = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True  # Only if your Ollama client supports this
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        # Adjust this depending on Ollama's streaming format
        delta = chunk.get("message", {}).get("content", "")
        response += delta or ''
        response = response.replace("```", "").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)
    with open("brochure.txt","w",encoding="utf-8") as f:
        f.write(response)

## Company Assistant using the Brochure Maker Bot

In [34]:
import gradio as gr

In [47]:
system_message = "You are a knowledgeable and professional assistant representing the company. A detailed brochure describing companies business ventures, services, history, and career opportunities has been provided to you in the conversation history.\
Your primary role is to answer user questions accurately and helpfully by referring to the brochure content. Always assume the brochure is the most reliable source of truth about the company.\
If a user asks about:\
- The company’s name, history, mission, or areas of expertise — respond clearly using the brochure.\
- Services or job opportunities — explain what our company offers, based on the brochure.\
- Business inquiries — respond warmly and professionally, offering a virtual tea or coffee as a gesture of hospitality.\
- Irrelevant or nonsensical questions — politely redirect the conversation to company-related topics.\
\
Do not invent facts. If the brochure does not contain the answer, say so clearly and offer to help with related information."


In [48]:
# create_stream_brochure("satyasai","https://www.satyasai.co.in")

In [49]:
with open("brochure.txt" ,"r") as f:
    brochure_text = f.read()

In [52]:
def chat(message,history):
    history = [{"role":h["role"],"content":h["content"]} for h in history]
    brochure_context = [{"role":"assistant","content":brochure_text}]
    messages = [{"role":"system","content":system_message}] + brochure_context + history + [{"role":"user","content":message}]
    stream = ollama.chat(model=MODEL,messages = messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk["message"]["content"]
        yield response

In [53]:
gr.ChatInterface(fn=chat,type='messages').launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
